# 05 · Audit of the s2cloudless cloud masks (Colab CPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1ow1atency/floodfusion/blob/main/notebooks/05_cloud_audit.ipynb)

Runs `scripts/audit_clouds.py` on every chip of all four splits (the ROADMAP Stage 5 TODO, findings in `docs/cloud_audit.md`):
- **Thresholds:** cloud fraction at s2cloudless thresholds 0.4 (current), 0.5 and 0.6, and the resulting real-cloud bin counts for test and bolivia.
- **Label check:** how the hand labels (-1 / land / water) split between cloudy and clear pixels.
- **Agreement:** how well each mask matches the annotators' no-data, on train chips only.
- **Probability maps:** figures of the 6 cloudiest and 6 mid-cloud train chips, plus Paraguay_215904 and Paraguay_36146.

It does not change `cloud_cover.csv`. Results go to `floodfusion/results/cloud_audit/` on Drive.

**CPU runtime is enough:** s2cloudless (LightGBM) does not use the GPU. Expect roughly 35-50 min for ~446 chips on Colab's 2 vCPUs. Needs `00_download_data.ipynb` to have run first.

Settings.

In [ ]:
DATA_ROOT = '/content/drive/MyDrive/floodfusion/data/sen1floods11'
OUT_DIR = '/content/drive/MyDrive/floodfusion/results/cloud_audit'  # chip_audit.csv, summary.md, figures, prob_maps/

Mount Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Clone the repo, or pull if it is already cloned. Uses the `GITHUB_TOKEN` Colab secret if set (see README "Colab setup"); otherwise tries a plain public clone.

In [ ]:
# Clone or update the repo in Colab. Shared cell: paste verbatim into every notebook
# (tests/test_notebooks.py checks the copies match notebooks/colab_clone_cell.py).
# Uses the Colab secret GITHUB_TOKEN if available (see README "Colab setup"). The token
# reaches git only through environment variables: never on the command line, never in
# .git/config (the remote URL stays token-free), and it is masked in any output.
import base64, os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/1ow1atency/floodfusion.git'
REPO_DIR = Path('/content/floodfusion')


def _github_token():
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception as e:  # secret missing, notebook access off, or not in Colab
        print(f'GITHUB_TOKEN secret not available ({type(e).__name__}); trying a plain public clone.\n'
              'To add it: key icon in the left sidebar -> "Add new secret" -> name GITHUB_TOKEN, value = a\n'
              'GitHub token with read access to this repo -> switch "Notebook access" on -> re-run this cell.')
        return None
    return token.strip() or None


def _git(*args, token=None):
    env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}  # fail instead of waiting for a password
    secrets = []
    if token:
        b64 = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
        env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                   GIT_CONFIG_VALUE_0=f'Authorization: Basic {b64}')
        secrets = [token, b64]
    r = subprocess.run(['git', *args], env=env, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    for s in secrets:
        out = out.replace(s, '***')
    if r.returncode != 0:
        hint = '' if token else '\n(No token used. If the repo is not publicly visible, add the GITHUB_TOKEN secret.)'
        raise RuntimeError(f'git {args[0] if args[0] != "-C" else args[2]} failed:\n{out}{hint}')
    return out


def _clone_or_pull():
    version = tuple(int(x) for x in _git('--version').split()[2].split('.')[:2])
    if version < (2, 31):  # GIT_CONFIG_COUNT needs git >= 2.31
        raise RuntimeError(f'git >= 2.31 needed, found {version}')
    token = _github_token()
    if (REPO_DIR / '.git').exists():
        _git('-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPO_URL)  # keep the remote token-free
        print(_git('-C', str(REPO_DIR), 'pull', '--ff-only', token=token))
    else:
        print(_git('clone', REPO_URL, str(REPO_DIR), token=token))
    print('at', _git('-C', str(REPO_DIR), 'log', '--oneline', '-1'))


_clone_or_pull()

Install the package with the `clouds` extra (s2cloudless).

In [ ]:
!pip install -q -e "{REPO_DIR}[clouds]"
assert _exit_code == 0, 'pip install failed, see output above'

Run the audit. No `--allow-missing`: on the full data any missing file must be an error. It also checks that threshold 0.4 reproduces `docs/cloud_cover.csv`.

In [ ]:
import time

t0 = time.time()
!cd {REPO_DIR} && python scripts/audit_clouds.py --root "{DATA_ROOT}" --out "{OUT_DIR}" --cloud-csv docs/cloud_cover.csv
assert _exit_code == 0, 'audit_clouds.py failed, see output above'
print(f'done in {(time.time() - t0) / 60:.1f} min')

Summary and figures (also saved in the results folder).

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

display(Markdown((Path(OUT_DIR) / 'summary.md').read_text()))
for name in ['prob_maps_cloudiest.png', 'prob_maps_mid_cloud.png']:
    print(name)
    display(Image(filename=str(Path(OUT_DIR) / name)))